# QA/QC: Pipeline Output Inspection

Scan pipeline output directory for CSV and NetCDF files produced by the incremental pipeline.

- **Topography** (CSV): plot each variable as a choropleth
- **Soils** (CSV): plot continuous soil properties + categorical texture
- **Land cover** (CSV): extract majority class per HRU and plot; continuous props as choropleths
- **Snow** (CSV): categorical CV_INT classification
- **Climate** (NetCDF): plot the last time-step for each variable

Works with both the DRB 2-year pipeline output (`output/`) and the GFv1.1 pw-check output (`pw-check/output/`).

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
# --- Configuration: set these paths for your run ---
# Option A: DRB 2-year pipeline output
# OUTPUT_DIR = Path("../output")
# FABRIC_PATH = Path("../data/pywatershed_gis/drb_2yr/nhru.gpkg")
# ID_FIELD = "nhm_id"

# Option B: GFv1.1 pw-check output
OUTPUT_DIR = Path("../pw-check/output")
FABRIC_PATH = Path("../data/pywatershed_gis/drb_2yr/nhru.gpkg")
ID_FIELD = "nhm_id"

fabric = gpd.read_file(FABRIC_PATH)
print(f"Fabric: {len(fabric)} features, CRS={fabric.crs}")
print(f"Output: {OUTPUT_DIR.resolve()}")

# NLCD class labels for majority-class plots
NLCD_LABELS = {
    11: "Open Water",
    12: "Perennial Ice/Snow",
    21: "Developed, Open",
    22: "Developed, Low",
    23: "Developed, Med",
    24: "Developed, High",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    51: "Dwarf Scrub",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    72: "Sedge/Herbaceous",
    73: "Lichens",
    74: "Moss",
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
}

# GFv1.1 PRMS cover type labels (NALCMS → PRMS classification)
PRMS_COV_TYPE = {
    0: "Bare soil",
    1: "Grasses",
    2: "Shrubs",
    3: "Trees",
    4: "Coniferous",
}

# GFv1.1 PRMS soil texture labels
PRMS_SOIL_TYPE = {
    1: "Sand",
    2: "Loam",
    3: "Clay",
}

In [ ]:
csv_files = sorted(OUTPUT_DIR.rglob("*.csv"))
nc_files = sorted(OUTPUT_DIR.rglob("*.nc"))

print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  {f.relative_to(OUTPUT_DIR)}")
print(f"\nFound {len(nc_files)} NetCDF files:")
for f in nc_files:
    print(f"  {f.relative_to(OUTPUT_DIR)}")

In [ ]:
def join_series_to_fabric(fabric, nhm_ids, values, id_field=ID_FIELD):
    """Join a series of values keyed by id_field to the fabric GeoDataFrame."""
    lookup = dict(zip(nhm_ids, values, strict=False))
    gdf = fabric.copy()
    gdf["_val"] = gdf[id_field].map(lookup)
    return gdf.dropna(subset=["_val"])


def plot_choropleth(gdf, title, ax, categorical=False, cmap="viridis", legend_kwds=None):
    """Plot a choropleth map on the given axes."""
    if gdf.empty or gdf["_val"].isna().all():
        ax.set_title(f"{title}\n(no data)")
        ax.set_axis_off()
        return
    if legend_kwds is None:
        legend_kwds = {"shrink": 0.6} if not categorical else {"loc": "lower left", "fontsize": 6}
    gdf.plot(
        column="_val",
        ax=ax,
        legend=True,
        cmap=cmap,
        categorical=categorical,
        legend_kwds=legend_kwds,
    )
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()


def plot_csv_choropleths(csv_files, section_title, categorical_vars=None):
    """Plot choropleths for a list of CSV files.

    Parameters
    ----------
    csv_files : list[Path]
        CSV files with columns [id_field, variable_name].
    section_title : str
        Title for the figure.
    categorical_vars : dict[str, dict] or None
        Map of variable names to label dicts for categorical plots.
        If None, all variables plotted as continuous.
    """
    if not csv_files:
        print(f"No {section_title} CSVs found.")
        return

    if categorical_vars is None:
        categorical_vars = {}

    ncols = min(3, len(csv_files))
    nrows = (len(csv_files) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for i, csv_path in enumerate(csv_files):
        df = pd.read_csv(csv_path)
        id_col = ID_FIELD if ID_FIELD in df.columns else df.columns[0]
        var_cols = [c for c in df.columns if c != id_col and not c.endswith("_count")]

        # Detect if this is a fraction/categorical CSV (many class columns)
        class_cols = [c for c in var_cols if c.rsplit("_", 1)[-1].isdigit()]

        if len(class_cols) > 3:
            # Categorical: extract majority class (guard against all-zero rows)
            class_codes = [int(c.rsplit("_", 1)[-1]) for c in class_cols]
            fractions = df[class_cols].values
            row_sums = np.nansum(fractions, axis=1)
            has_data = row_sums > 0
            majority_codes = np.full(len(fractions), np.nan)
            if has_data.any():
                majority_codes[has_data] = np.array(class_codes)[
                    np.argmax(fractions[has_data], axis=1)
                ]

            labels = {**NLCD_LABELS, **PRMS_COV_TYPE}
            majority_labels = [
                labels.get(int(c), str(int(c))) if np.isfinite(c) else "no data"
                for c in majority_codes
            ]
            gdf = join_series_to_fabric(fabric, df[id_col], majority_labels)
            plot_choropleth(gdf, f"{csv_path.stem} (majority)", axes[i],
                           categorical=True, cmap="tab20")
        elif len(var_cols) == 1:
            var_name = var_cols[0]
            # Check if categorical
            if var_name in categorical_vars:
                labels = categorical_vars[var_name]
                mapped = []
                for v in df[var_name]:
                    try:
                        mapped.append(labels.get(int(v), str(int(v))) if np.isfinite(v) else "")
                    except (TypeError, ValueError):
                        mapped.append("")
                gdf = join_series_to_fabric(fabric, df[id_col], mapped)
                plot_choropleth(gdf, var_name, axes[i], categorical=True, cmap="tab10")
            else:
                gdf = join_series_to_fabric(fabric, df[id_col], df[var_name])
                plot_choropleth(gdf, var_name, axes[i])
        else:
            # Multiple continuous columns — plot first one
            var_name = var_cols[0]
            gdf = join_series_to_fabric(fabric, df[id_col], df[var_name])
            plot_choropleth(gdf, f"{csv_path.stem}: {var_name}", axes[i])

    for j in range(len(csv_files), len(axes)):
        axes[j].set_visible(False)
    fig.suptitle(section_title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## Topography (static CSVs)

Each CSV has columns `nhm_id, <variable>`. Plot as continuous choropleths.

In [ ]:
topo_dir = OUTPUT_DIR / "topography"
topo_files = sorted(topo_dir.glob("*.csv")) if topo_dir.exists() else []
plot_csv_choropleths(topo_files, "Topography")

## Soils (CSVs)

Continuous soil properties (sand/silt/clay percentages, AWC, ksat, etc.)
and categorical PRMS soil texture class.

In [ ]:
soils_dir = OUTPUT_DIR / "soils"
soils_files = sorted(soils_dir.glob("*.csv")) if soils_dir.exists() else []
plot_csv_choropleths(
    soils_files,
    "Soils",
    categorical_vars={"soil_type": PRMS_SOIL_TYPE},
)

## Land Cover (CSVs)

Categorical land cover (NLCD class fractions or PRMS cov_type) and continuous
properties (impervious fraction, canopy cover, interception, root depth, cover density).

In [ ]:
lc_dir = OUTPUT_DIR / "land_cover"
lc_files = sorted(lc_dir.glob("*.csv")) if lc_dir.exists() else []
plot_csv_choropleths(
    lc_files,
    "Land Cover",
    categorical_vars={"cov_type": PRMS_COV_TYPE},
)

## Snow (CSVs)

GFv1.1 CV_INT classification values for snow depletion curves.

In [ ]:
snow_dir = OUTPUT_DIR / "snow"
snow_csv_files = sorted(snow_dir.glob("*.csv")) if snow_dir.exists() else []
plot_csv_choropleths(snow_csv_files, "Snow")

## Climate (temporal NetCDF)

Each NetCDF has dimensions `(time, nhm_id)`. Plot the last time-step for each variable.

In [ ]:
climate_dir = OUTPUT_DIR / "climate"
climate_files = sorted(climate_dir.glob("*.nc")) if climate_dir.exists() else []

for nc_path in climate_files:
    ds = xr.open_dataset(nc_path)
    last_time = pd.Timestamp(ds.time.values[-1]).strftime("%Y-%m-%d")
    ds_last = ds.isel(time=-1)

    # ID dimension — find spatial dimension (not time)
    non_time_dims = list(ds.dims.keys() - {"time"})
    if not non_time_dims:
        print(f"Skipping {nc_path.name}: no spatial dimension (dims={list(ds.dims)})")
        ds.close()
        continue
    id_dim = "nhm_id" if "nhm_id" in ds.dims else non_time_dims[0]
    data_vars = [v for v in ds_last.data_vars if id_dim in ds_last[v].dims]

    ncols = min(3, len(data_vars))
    nrows = (len(data_vars) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for i, var_name in enumerate(data_vars):
        da = ds_last[var_name]
        gdf = join_series_to_fabric(fabric, da[id_dim].values, da.values)
        plot_choropleth(gdf, var_name, axes[i])

    for j in range(len(data_vars), len(axes)):
        axes[j].set_visible(False)

    rel_path = nc_path.relative_to(OUTPUT_DIR)
    fig.suptitle(f"{rel_path} (last step: {last_time})", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
    ds.close()

if not climate_files:
    print("No climate NetCDF files found.")